## My Final Project

You can use this notebook if you'd like for your final project. Or you can copy a notebook from one of our lessons. Or make your own! 

## Pitch



#This is the scraper I ran to download pdfs directly from the database.
#It took three tries before Claude produced a code that worked. The download buttrons run JavaScript and do not
#link directly to a PDF.

"""
Download PDFs from the NYC 9/11 Document Portal for one search phrase (version 3).

The portal's "Download" buttons are not plain links to PDF files. They are
buttons that run JavaScript. So this script clicks each result's Download
button and saves the file the browser receives.

Version 3 finds the Download buttons in a more flexible way than version 2,
and saves a diagnostic file (download_elements.json) showing what it found.

Run:
    python portal_download_v3.py
"""

import csv
import json
import re
import sys
import time
import zipfile
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

from playwright.sync_api import sync_playwright, TimeoutError as PWTimeout

# ----------------------------- settings you can change -----------------------------
PORTAL_URL = "https://sept11documents.cityofnewyork.us/"
SEARCH_TERM = "cleaning request"
MAX_PAGES = 3             # how many pages of search results to download from
DELAY_SECONDS = 3.0       # pause between downloads, to be polite to the server
MAX_FAILURES_IN_A_ROW = 3 # stop early if this many downloads in a row fail
OUTPUT_DIR = Path("portal_output_v3")
# ------------------------------------------------------------------------------------

DOC_ID_RE = re.compile(r"NYC-WTC_\d+")
DOWNLOAD_TEXT_RE = re.compile(r"^\s*Download\s*$", re.I)

SEARCH_SELECTORS = [
    "input[type=search]",
    "input[placeholder*='earch' i]",
    "input[aria-label*='earch' i]",
    "input[name*='earch' i]",
    "input[id*='earch' i]",
    "input[type=text]",
]

# Buttons that might appear in a pop-up asking you to confirm a download.
CONFIRM_RE = re.compile(r"^\s*(ok|okay|continue|accept|i agree|agree|proceed|confirm|yes)\s*$", re.I)
DIALOG_CONFIRM_RE = re.compile(
    r"^\s*(ok|okay|continue|accept|i agree|agree|proceed|confirm|yes|download)\s*$", re.I)

# Runs in the browser: finds which result a Download button belongs to.
CONTEXT_JS = """
(el, pattern) => {
  const reAll = new RegExp(pattern, 'g');
  const clean = t => t.replace(/\\s+/g, ' ').slice(0, 500);
  let node = el;
  let fallback = null;
  for (let i = 0; i < 15 && node; i++) {
    const text = (node.innerText || '').trim();
    const found = text.match(reAll);
    if (found && found.length === 1) {
      return { id: found[0], text: clean(text) };
    }
    if (found && found.length > 1) { break; }
    if (!fallback && text.length > 60 && text.length < 1500) { fallback = text; }
    node = node.parentElement || (node.getRootNode && node.getRootNode().host) || null;
  }
  return { id: '', text: fallback ? clean(fallback) : '' };
}
"""

# Runs in the browser: lists every element whose text is exactly "Download".
DUMP_JS = """
els => els
  .filter(e => (e.textContent || '').trim() === 'Download' && (e.childElementCount === 0 || e.tagName === 'A'))
  .slice(0, 30)
  .map(e => ({
    tag: e.tagName,
    class: (e.className && e.className.toString) ? e.className.toString() : '',
    role: e.getAttribute('role'),
    aria_label: e.getAttribute('aria-label'),
    title: e.getAttribute('title'),
    href: e.getAttribute('href'),
    visible: e.offsetParent !== null
  }))
"""


def check_robots():
    """Stop if the site asks automated tools to stay out."""
    rp = RobotFileParser()
    rp.set_url(urljoin(PORTAL_URL, "/robots.txt"))
    try:
        rp.read()
    except Exception as exc:
        print(f"Could not read robots.txt ({exc}).")
        print("Continuing, but please check the portal's terms of use yourself.")
        return
    if not rp.can_fetch("*", PORTAL_URL):
        print("The site's robots.txt asks automated tools not to access it.")
        print("Stopping. Please contact the city (NYC Law Department) about bulk access.")
        sys.exit(1)
    print("robots.txt allows access to the portal home page.")


def settle(page, extra_ms=2500):
    """Wait for the page to finish loading its content."""
    try:
        page.wait_for_load_state("networkidle", timeout=20000)
    except Exception:
        pass
    page.wait_for_timeout(extra_ms)


def first_visible(locator):
    """Return the first visible match of a locator, or None."""
    try:
        for i in range(locator.count()):
            item = locator.nth(i)
            if item.is_visible():
                return item
    except Exception:
        pass
    return None


def find_search_box(page):
    for selector in SEARCH_SELECTORS:
        try:
            box = first_visible(page.locator(selector))
            if box is not None:
                return box
        except Exception:
            continue
    return None


def run_search(page):
    box = find_search_box(page)
    if box is not None:
        print(f'Searching for "{SEARCH_TERM}" ...')
        box.click()
        box.fill(SEARCH_TERM)
        box.press("Enter")
        settle(page, 4000)
    else:
        print("I could not find the search box automatically.")
        print(f'In the browser window, type "{SEARCH_TERM}" into the search box and run the search.')
        input("When results are showing, press Enter here to continue... ")
        settle(page)


def download_buttons(page):
    """Find the Download buttons, trying several ways."""
    options = [
        page.locator("a", has_text=DOWNLOAD_TEXT_RE),
        page.get_by_text("Download", exact=True),
        page.locator("button, [role=button]", has_text=DOWNLOAD_TEXT_RE),
    ]
    for loc in options:
        try:
            if first_visible(loc) is not None:
                return loc
        except Exception:
            continue
    return options[0]


def describe_button(button):
    """Return {'id': ..., 'text': ...} for the result this Download button belongs to."""
    try:
        info = button.evaluate(CONTEXT_JS, DOC_ID_RE.pattern)
        if info:
            return {"id": info.get("id", ""), "text": info.get("text", "")}
    except Exception:
        pass
    return {"id": "", "text": ""}


def first_result_key(page):
    """A short fingerprint of the first result on the page (used to tell when the page changed)."""
    button = first_visible(download_buttons(page))
    if button is None:
        return None
    info = describe_button(button)
    return info["id"] or info["text"][:80] or "unknown"


def wait_for_results(page, timeout_s=30):
    end = time.time() + timeout_s
    while time.time() < end:
        if first_result_key(page):
            return True
        page.wait_for_timeout(500)
    return False


def results_on_page(page):
    """List the results on the current page: [(button, id, text)], without duplicates."""
    buttons = download_buttons(page)
    found, seen = [], set()
    for i in range(buttons.count()):
        button = buttons.nth(i)
        try:
            if not button.is_visible():
                continue
        except Exception:
            continue
        info = describe_button(button)
        key = info["id"] or info["text"][:80] or f"unknown_{i}"
        if key in seen:
            continue
        seen.add(key)
        found.append((button, info["id"], info["text"]))
    return found


def go_to_next_page(page, current_page, old_key):
    """Move to the next page of results. Returns True if the results changed."""
    n = str(current_page + 1)
    candidates = [
        page.locator("a", has_text=re.compile(rf"^\s*{n}\s*$")),
        page.get_by_role("link", name=n, exact=True),
        page.locator("a[rel=next]"),
        page.locator("a", has_text=re.compile(r"^\s*(next|\u00bb|>)\s*$", re.I)),
    ]
    for loc in candidates:
        link = first_visible(loc)
        if link is None:
            continue
        try:
            link.click()
            end = time.time() + 20
            while time.time() < end:
                new_key = first_result_key(page)
                if new_key and new_key != old_key:
                    settle(page, 1500)
                    return True
                page.wait_for_timeout(500)
        except Exception:
            continue
        break  # clicked but nothing changed: ask the person instead of guessing

    print("I could not move to the next page automatically.")
    answer = input(
        f"In the browser window, go to results page {current_page + 1} yourself, then press Enter "
        "(or type stop and press Enter to finish): "
    )
    if answer.strip().lower() == "stop":
        return False
    settle(page)
    return True


def find_confirm_button(page):
    """Look for a pop-up 'OK / Continue / Accept' type button."""
    dialogs = page.locator("[role=dialog], [role=alertdialog], dialog, .modal")
    for scope, pattern in ((dialogs, DIALOG_CONFIRM_RE), (page, CONFIRM_RE)):
        for role in ("button", "link"):
            try:
                loc = scope.get_by_role(role, name=pattern)
                candidate = first_visible(loc)
                if candidate is not None:
                    return candidate
            except Exception:
                continue
    return None


def try_download(page, context, button, new_pages):
    """
    Click one Download button. Returns (pdf_bytes, suggested_name, note).
    pdf_bytes is None if it did not work, and note says why.
    """
    pages_before = len(new_pages)

    # 1. The normal case: clicking starts a file download.
    try:
        with page.expect_download(timeout=20000) as dl_info:
            button.click(timeout=15000)
        download = dl_info.value
        return Path(download.path()).read_bytes(), download.suggested_filename, "ok"
    except PWTimeout:
        pass
    except Exception as exc:
        return None, "", f"click failed: {exc}"

    # 2. Maybe a pop-up asked for confirmation first.
    confirm = find_confirm_button(page)
    if confirm is not None:
        try:
            with page.expect_download(timeout=30000) as dl_info:
                confirm.click(timeout=15000)
            download = dl_info.value
            return Path(download.path()).read_bytes(), download.suggested_filename, "ok (after confirm)"
        except PWTimeout:
            pass
        except Exception as exc:
            return None, "", f"confirm click failed: {exc}"

    # 3. Maybe the PDF opened in a new browser tab instead.
    for new_page in new_pages[pages_before:]:
        try:
            new_page.wait_for_load_state("domcontentloaded", timeout=15000)
            url = new_page.url
            new_page.close()
            if url.startswith("http"):
                resp = context.request.get(url, timeout=90000)
                body = resp.body()
                if body[:5] == b"%PDF-":
                    return body, "", "ok (opened in new tab)"
        except Exception:
            continue

    return None, "", "no download started after clicking"


def save_problem_notes(page, debug_dir, label):
    """Save a screenshot, the visible button text, and a list of Download elements."""
    try:
        page.screenshot(path=str(debug_dir / f"problem_{label}.png"), full_page=True)
    except Exception:
        pass
    try:
        texts = page.eval_on_selector_all(
            "button, a[href], [role=button]",
            "els => els.filter(e => e.offsetParent !== null)"
            ".map(e => (e.innerText || e.getAttribute('aria-label') || '').trim())"
            ".filter(t => t).slice(0, 80)",
        )
        (debug_dir / f"problem_{label}.txt").write_text("\n".join(texts), encoding="utf-8")
    except Exception:
        pass
    try:
        rows = page.eval_on_selector_all("*", DUMP_JS)
        (debug_dir / f"download_elements_{label}.json").write_text(
            json.dumps(rows, indent=2), encoding="utf-8")
    except Exception:
        pass


def write_csv(path, rows, fieldnames):
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def clean_filename(name):
    return re.sub(r'[<>:"/\\|?*\x00-\x1f]', "_", name)


def main():
    check_robots()

    pdf_dir = OUTPUT_DIR / "pdfs"
    debug_dir = OUTPUT_DIR / "debug"
    pdf_dir.mkdir(parents=True, exist_ok=True)
    debug_dir.mkdir(parents=True, exist_ok=True)

    manifest = []
    saved_ids = set()
    failures_in_a_row = 0
    problem_count = 0
    stop_everything = False

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        context = browser.new_context(accept_downloads=True)
        new_pages = []
        context.on("page", lambda pg: new_pages.append(pg))
        page = context.new_page()

        print(f"Opening {PORTAL_URL}")
        page.goto(PORTAL_URL, wait_until="load", timeout=60000)
        settle(page)

        run_search(page)
        if not wait_for_results(page):
            print("No results appeared. Saving a screenshot and notes for diagnosis.")
            save_problem_notes(page, debug_dir, "no_results")

        for page_no in range(1, MAX_PAGES + 1):
            settle(page, 1500)
            page.screenshot(path=str(debug_dir / f"results_page{page_no}.png"), full_page=True)
            results = results_on_page(page)
            print(f"\nResults page {page_no}: {len(results)} result(s) found.")
            if not results:
                save_problem_notes(page, debug_dir, f"page{page_no}_empty")

            for number, (button, doc_id, text) in enumerate(results, start=1):
                label = doc_id or f"page{page_no}_item{number}"
                if doc_id and doc_id in saved_ids:
                    continue

                body, suggested, note = try_download(page, context, button, new_pages)

                filename = ""
                status = note
                if body is not None and body[:5] == b"%PDF-":
                    filename = clean_filename(f"{doc_id}.pdf" if doc_id else (suggested or f"{label}.pdf"))
                    (pdf_dir / filename).write_bytes(body)
                    if doc_id:
                        saved_ids.add(doc_id)
                    failures_in_a_row = 0
                else:
                    if body is not None:
                        status = "downloaded file was not a PDF"
                    failures_in_a_row += 1
                    problem_count += 1
                    save_problem_notes(page, debug_dir, f"{problem_count}")
                    try:
                        page.keyboard.press("Escape")  # close any pop-up
                    except Exception:
                        pass

                print(f"  [{page_no}.{number}] {label}: {status}")
                manifest.append({
                    "search_term": SEARCH_TERM,
                    "results_page": page_no,
                    "document_id": doc_id,
                    "result_text": text,
                    "local_file": filename,
                    "status": status,
                    "downloaded_at": datetime.now().isoformat(timespec="seconds"),
                })

                if failures_in_a_row >= MAX_FAILURES_IN_A_ROW:
                    print(f"\n{MAX_FAILURES_IN_A_ROW} downloads in a row failed, so I am stopping.")
                    stop_everything = True
                    break
                time.sleep(DELAY_SECONDS)

            if stop_everything or page_no == MAX_PAGES or not results:
                break
            if not go_to_next_page(page, page_no, first_result_key(page)):
                print("Stopping page collection.")
                break

        browser.close()

    manifest_path = OUTPUT_DIR / "manifest.csv"
    write_csv(manifest_path, manifest,
              ["search_term", "results_page", "document_id", "result_text",
               "local_file", "status", "downloaded_at"])

    zip_path = OUTPUT_DIR / "cleaning_request_documents.zip"
    downloaded = [m for m in manifest if m["local_file"]]
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(manifest_path, "manifest.csv")
        for m in downloaded:
            zf.write(pdf_dir / m["local_file"], f"pdfs/{m['local_file']}")

    print(f"\nDone. {len(downloaded)} of {len(manifest)} file(s) downloaded.")
    print(f"ZIP file: {zip_path.resolve()}")
    if len(downloaded) < max(len(manifest), 1):
        print("\nSome downloads did not work. Please send the files in this folder:")
        print(f"  {debug_dir.resolve()}")
        print("(the .png, .txt and download_elements .json files are the most useful).")


if __name__ == "__main__":
    main()